SETUP

In [138]:
import pandas as pd
import sqlite3
import openpyxl

In [139]:
mer_df = pd.read_csv("csv_file/merchants.csv")
refund_df = pd.read_csv("csv_file/refunds.csv")
tran_info_df = pd.read_csv("csv_file/transaction_info.csv")
tran_df = pd.read_csv("csv_file/transactions.csv")

In [140]:
display(mer_df[:1])
display(refund_df[:1])
display(tran_info_df[:1])
display(tran_df[:1])

,merchant_id,leader_name,create_date,address,country,email,phone,industry
0,Richards-Little,Ms. Rachel Baker,2025-10-23,"7251 Patterson Station, New Jeremyshire, IL 72094",China,maciasandrew@simpson.info,206-857-7069x1009,Food & Beverage


,refund_id,tran_id,channel,amount,create_date,refund_date,reason,status
0,d0c37a23-9663-4410-89d3-62cfe87bb5e0,4a8c5a13-45de-451d-98ba-711f2cd9650a,DuitNow QR,45.74,2025-06-18 10:55:17,2025-06-25 10:55:17,Customer Dissatisfaction,completed


,tran_id,country,card_type,card_brands,region,channel_type
0,377a06dd-16de-4064-a464-da56c85039c8,Malaysia,NaN,NaN,Domestic,Buy Now Pay Later


,tran_id,merchant_id,channel,amount,currency,date,status
0,377a06dd-16de-4064-a464-da56c85039c8,"Townsend, Bishop and Mccoy",Shopee PayLater,46.95,MYR,2025-12-05 11:31:17,completed


In [141]:
# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")

# Load DataFrames into SQL tables
mer_df.to_sql("merchants", conn, index=False, if_exists='replace')
refund_df.to_sql("refunds", conn, index=False, if_exists='replace')
tran_info_df.to_sql("transaction_info", conn, index=False, if_exists='replace')
tran_df.to_sql("transactions", conn, index=False, if_exists='replace')

545668

GLOBAL FUNCTION

QUERY

In [142]:
sqlInput = f"""
SELECT t.merchant_id, COUNT(t.tran_id) as count
FROM transactions t
JOIN merchants m ON t.merchant_id = m.merchant_id
WHERE m.country = 'Singapore'
AND date BETWEEN '2025-12-31 00:00:00' AND '2025-12-31 23:59:59'
GROUP BY t.merchant_id
ORDER BY count DESC
"""

txn = pd.read_sql_query(sqlInput, conn)

display(txn[:10])

,merchant_id,count
0,"Townsend, Bishop and Mccoy",9
1,"Roman, Willis and Rojas",5
2,Owen Group,5
3,Vega-Nelson,4
4,"Romero, Morrison and Morales",4
5,"Ramirez, Cruz and Patrick",4
6,Mejia and Sons,4
7,"Smith, Cooper and Cabrera",2
8,Dixon-Johnson,2
9,Acosta PLC,2


In [143]:
merchant_id = txn['merchant_id'][0]
merchant_id

'Townsend, Bishop and Mccoy'

In [144]:
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
transactions = []

for _ in range(33):
    tran_id = fake.uuid4()
    
    years = [2022, 2023, 2024, 2025]
    year_weights = [1, 1.2, 1.8, 1.5]  

    year = random.choices(years, weights=year_weights)[0]

    start = datetime(2025, 12, 31, 11, 30, 00)
    end = datetime(2025, 12, 31, 11, 59, 59)

    trans_date = fake.date_time_between(start_date=start, end_date=end)

    amount_sgd = round(random.triangular(1, 30, 10), 2)

    transactions.append({
        'tran_id': tran_id,
        'merchant_id': merchant_id,
        'channel': 'DuitNow QR Offline',
        'amount': amount_sgd,
        'currency': 'SGD',
        'date': trans_date,
        'status': 'cancelled'
    })

cancelled_txn_df = pd.DataFrame(transactions)

display(cancelled_txn_df)

,tran_id,merchant_id,channel,amount,currency,date,status
0,418b2e81-ff57-4800-887f-fb8c8a9fdb72,"Townsend, Bishop and Mccoy",DuitNow QR Offline,10.08,SGD,2025-12-31 11:54:03,cancelled
1,91e63aa0-5efd-47c4-95d8-81808fe43aea,"Townsend, Bishop and Mccoy",DuitNow QR Offline,12.82,SGD,2025-12-31 11:56:38,cancelled
2,aabb5506-d546-46a0-82f1-8fe7390d4d51,"Townsend, Bishop and Mccoy",DuitNow QR Offline,5.40,SGD,2025-12-31 11:52:48,cancelled
3,95523b10-ee28-4b1b-a0b9-aedd8a6358da,"Townsend, Bishop and Mccoy",DuitNow QR Offline,17.08,SGD,2025-12-31 11:51:35,cancelled
4,2eccc13d-72a1-4fa5-a949-abdfd5a51bd7,"Townsend, Bishop and Mccoy",DuitNow QR Offline,17.92,SGD,2025-12-31 11:46:19,cancelled
5,c42144ff-728a-42f7-b4d2-07de5adf05b9,"Townsend, Bishop and Mccoy",DuitNow QR Offline,22.87,SGD,2025-12-31 11:49:58,cancelled
6,a4cf786a-76d7-4d06-8d59-ff6b6deec5ea,"Townsend, Bishop and Mccoy",DuitNow QR Offline,24.79,SGD,2025-12-31 11:30:04,cancelled
7,3fd53924-e5ce-4cb5-845b-9b32ef008fa3,"Townsend, Bishop and Mccoy",DuitNow QR Offline,8.77,SGD,2025-12-31 11:41:34,cancelled
8,c6e3c50e-7fb8-490a-b5f1-24e2459cf31b,"Townsend, Bishop and Mccoy",DuitNow QR Offline,20.78,SGD,2025-12-31 11:58:04,cancelled
9,ebfdaf93-8458-468d-984a-e1e6df232a8b,"Townsend, Bishop and Mccoy",DuitNow QR Offline,10.76,SGD,2025-12-31 11:33:55,cancelled


In [145]:
refunds = []

refund_reason_weights = {
    'Product Defect': 0.25,
    'Wrong Item Sent': 0.3,
    'Customer Dissatisfaction': 0.1,
    'Delayed Delivery': 0.15,
    'Fraudulent Transaction': 0.05,
    'Other': 0.15
}

for _, row in cancelled_txn_df.iterrows():

    tran_id = row['tran_id']
    tran_amount = row['amount']
    tran_date = row['date']
    channel = row['channel']

    channel_type = 'E-wallet Offline'
    status = 'completed'
    create_date = tran_date
    refund_date = tran_date
    refund_amount = tran_amount

    refunds.append({
        'refund_id': fake.uuid4(),
        'tran_id': tran_id,
        'channel': channel,
        'amount': refund_amount,
        'create_date': create_date,
        'refund_date': refund_date,
        'reason': random.choices(
            list(refund_reason_weights.keys()),
            weights=refund_reason_weights.values()
        )[0],
        'status': status
    })

cancelled_refunds_df = pd.DataFrame(refunds)

display(cancelled_refunds_df)

,refund_id,tran_id,channel,amount,create_date,refund_date,reason,status
0,447842a0-1efa-47ca-8b0a-aabca100b232,418b2e81-ff57-4800-887f-fb8c8a9fdb72,DuitNow QR Offline,10.08,2025-12-31 11:54:03,2025-12-31 11:54:03,Customer Dissatisfaction,completed
1,8da38d85-dd97-4cdc-98ce-49952e3eb715,91e63aa0-5efd-47c4-95d8-81808fe43aea,DuitNow QR Offline,12.82,2025-12-31 11:56:38,2025-12-31 11:56:38,Delayed Delivery,completed
2,33acc64e-3185-46d3-8972-f0663e621dbf,aabb5506-d546-46a0-82f1-8fe7390d4d51,DuitNow QR Offline,5.40,2025-12-31 11:52:48,2025-12-31 11:52:48,Product Defect,completed
3,74d4e4a0-b13b-49de-a56d-8cdaf708189f,95523b10-ee28-4b1b-a0b9-aedd8a6358da,DuitNow QR Offline,17.08,2025-12-31 11:51:35,2025-12-31 11:51:35,Delayed Delivery,completed
4,8856b62b-2677-42ee-8eab-02025bb8456b,2eccc13d-72a1-4fa5-a949-abdfd5a51bd7,DuitNow QR Offline,17.92,2025-12-31 11:46:19,2025-12-31 11:46:19,Product Defect,completed
5,03920a94-443b-4857-996a-ee5431040c72,c42144ff-728a-42f7-b4d2-07de5adf05b9,DuitNow QR Offline,22.87,2025-12-31 11:49:58,2025-12-31 11:49:58,Wrong Item Sent,completed
6,861ad533-c840-47fc-8a1f-4e5e73a5c108,a4cf786a-76d7-4d06-8d59-ff6b6deec5ea,DuitNow QR Offline,24.79,2025-12-31 11:30:04,2025-12-31 11:30:04,Wrong Item Sent,completed
7,82f77370-9e2d-400e-8eec-7831ad07e0fb,3fd53924-e5ce-4cb5-845b-9b32ef008fa3,DuitNow QR Offline,8.77,2025-12-31 11:41:34,2025-12-31 11:41:34,Delayed Delivery,completed
8,385edeed-7596-4281-98ed-bc4497364917,c6e3c50e-7fb8-490a-b5f1-24e2459cf31b,DuitNow QR Offline,20.78,2025-12-31 11:58:04,2025-12-31 11:58:04,Delayed Delivery,completed
9,f37d7332-77ce-4948-8441-45097dcfaa0f,ebfdaf93-8458-468d-984a-e1e6df232a8b,DuitNow QR Offline,10.76,2025-12-31 11:33:55,2025-12-31 11:33:55,Wrong Item Sent,completed


In [146]:
transaction_info = []

for _, row in cancelled_txn_df.iterrows():
    region = "Foreign"
    channel_type = 'E-wallet Offline'

    card_brands, card_type = None, None
    
    country = 'Singapore'

    transaction_info.append({
        'tran_id': row['tran_id'],
        'country': country,
        'card_type': card_type,
        'card_brands': card_brands,
        'region': region,
        'channel_type': channel_type
    })

cancelled_txn_info = pd.DataFrame(transaction_info)
display(cancelled_txn_info)

,tran_id,country,card_type,card_brands,region,channel_type
0,418b2e81-ff57-4800-887f-fb8c8a9fdb72,Singapore,None,None,Foreign,E-wallet Offline
1,91e63aa0-5efd-47c4-95d8-81808fe43aea,Singapore,None,None,Foreign,E-wallet Offline
2,aabb5506-d546-46a0-82f1-8fe7390d4d51,Singapore,None,None,Foreign,E-wallet Offline
3,95523b10-ee28-4b1b-a0b9-aedd8a6358da,Singapore,None,None,Foreign,E-wallet Offline
4,2eccc13d-72a1-4fa5-a949-abdfd5a51bd7,Singapore,None,None,Foreign,E-wallet Offline
5,c42144ff-728a-42f7-b4d2-07de5adf05b9,Singapore,None,None,Foreign,E-wallet Offline
6,a4cf786a-76d7-4d06-8d59-ff6b6deec5ea,Singapore,None,None,Foreign,E-wallet Offline
7,3fd53924-e5ce-4cb5-845b-9b32ef008fa3,Singapore,None,None,Foreign,E-wallet Offline
8,c6e3c50e-7fb8-490a-b5f1-24e2459cf31b,Singapore,None,None,Foreign,E-wallet Offline
9,ebfdaf93-8458-468d-984a-e1e6df232a8b,Singapore,None,None,Foreign,E-wallet Offline


In [147]:
updated_txns = pd.concat([tran_df, cancelled_txn_df])
updated_refunds = pd.concat([refund_df, cancelled_refunds_df])
updated_txn_info = pd.concat([tran_info_df, cancelled_txn_info])

output_folder = 'csv_file'

updated_txns.to_csv(f'{output_folder}/transactions.csv', index=False)
updated_refunds.to_csv(f'{output_folder}/refunds.csv', index=False)
updated_txn_info.to_csv(f'{output_folder}/transaction_info.csv', index=False)

In [148]:
mer_df = pd.read_csv("csv_file/merchants.csv")
refund_df = pd.read_csv("csv_file/refunds.csv")
tran_info_df = pd.read_csv("csv_file/transaction_info.csv")
tran_df = pd.read_csv("csv_file/transactions.csv")

conn = sqlite3.connect("payments.db")

mer_df.to_sql("merchants", conn, index=False, if_exists='replace')
refund_df.to_sql("refunds", conn, index=False, if_exists='replace')
tran_info_df.to_sql("transaction_info", conn, index=False, if_exists='replace')
tran_df.to_sql("transactions", conn, index=False, if_exists='replace')

545701

Checking

In [149]:
sqlInput = f"""
SELECT t.date, t.merchant_id, t.tran_id, t.status, t.currency,
    r.refund_id, r.status, r.create_date, r.refund_date
FROM transactions t
LEFT JOIN refunds r ON t.tran_id = r.tran_id
WHERE t.date BETWEEN '2025-12-31 00:00:00' AND '2025-12-31 23:59:59'
AND t.merchant_id = '{merchant_id}'
"""

txncheck = pd.read_sql_query(sqlInput, conn)

display(txncheck)

,date,merchant_id,tran_id,status,currency,refund_id,status,create_date,refund_date
0,2025-12-31 23:43:17,"Townsend, Bishop and Mccoy",2494b5c1-c6c6-42b2-b731-60bfc908be89,recorded,MYR,36f0a19a-2c3e-4adf-9bb4-26f71815c640,pending,2025-12-31 23:43:17,None
1,2025-12-31 09:17:08,"Townsend, Bishop and Mccoy",c1edd2f0-3213-41db-b772-48c0c2c92654,pending,THB,None,None,None,None
2,2025-12-31 06:47:16,"Townsend, Bishop and Mccoy",2e680961-1e70-4bfe-b042-619110c03fbe,pending,MYR,None,None,None,None
3,2025-12-31 12:14:52,"Townsend, Bishop and Mccoy",ad526eba-d789-4fdf-b73a-366e2f84ec3a,recorded,MYR,None,None,None,None
4,2025-12-31 08:13:51,"Townsend, Bishop and Mccoy",40611d2f-1c93-4c63-b00b-d032ecaeb4bb,pending,MYR,None,None,None,None
5,2025-12-31 03:56:49,"Townsend, Bishop and Mccoy",960b249d-68bf-474f-a751-fb8675352128,pending,SGD,None,None,None,None
6,2025-12-31 22:26:51,"Townsend, Bishop and Mccoy",3c7c3cd6-c442-4602-81f1-79e84ece1111,recorded,CNY,None,None,None,None
7,2025-12-31 17:37:52,"Townsend, Bishop and Mccoy",d0fad0e5-b62c-4632-ad9e-a0171c0325aa,pending,MYR,None,None,None,None
8,2025-12-31 17:16:00,"Townsend, Bishop and Mccoy",2cb78c75-deb8-4835-a44a-7a8dd1c4e925,pending,MYR,None,None,None,None
9,2025-12-31 11:54:03,"Townsend, Bishop and Mccoy",418b2e81-ff57-4800-887f-fb8c8a9fdb72,cancelled,SGD,447842a0-1efa-47ca-8b0a-aabca100b232,completed,2025-12-31 11:54:03,2025-12-31 11:54:03
